# Laboratory Session 4

In [1]:
import pandas as pd
import re
import nltk
import string
pd.options.mode.chained_assignment=None

In [2]:
full_df=pd.read_csv('bugtest.csv', nrows=30)
df=full_df[['text']]
df['text']=df['text'].astype(str)

In [3]:
df.head(10)

,text
0,user-ag mozilla/4 compat msie window nt avant ...
1,beta recreat shipped-local file browser scratc...
2,moment nscssframeconstructor attributechang ca...
3,user-ag mozilla/5 window window nt en-u rv gec...
4,upload extens wrong min/maxvers lengthi error ...
5,user-ag mozilla/4 compat msie window nt sv1 ne...
6,user-ag mozilla/5 x11 linux i686 en-gb rv geck...
7,user-ag mozilla/5 window window nt en-u rv b2 ...
8,user-ag mozilla/5 window window nt en-u rv a1 ...
9,testcas bug cairo build border border-radiu do...


### 1. Convert Into Lowercase Format

In [4]:
df['lowertext']=df['text'].str.lower()
df['lowertext'].head(10)

0    user-ag mozilla/4 compat msie window nt avant ...
1    beta recreat shipped-local file browser scratc...
2    moment nscssframeconstructor attributechang ca...
3    user-ag mozilla/5 window window nt en-u rv gec...
4    upload extens wrong min/maxvers lengthi error ...
5    user-ag mozilla/4 compat msie window nt sv1 ne...
6    user-ag mozilla/5 x11 linux i686 en-gb rv geck...
7    user-ag mozilla/5 window window nt en-u rv b2 ...
8    user-ag mozilla/5 window window nt en-u rv a1 ...
9    testcas bug cairo build border border-radiu do...
Name: lowertext, dtype: object

### 2. Remove Punctuation

In [5]:
PUNCT=string.punctuation
def remove_punct(text):
    return text.translate(str.maketrans('', '', PUNCT))
df['nopuntext']=df['text'].apply(lambda text: remove_punct(text))
df['nopuntext'].head(10)

0    userag mozilla4 compat msie window nt avant br...
1    beta recreat shippedlocal file browser scratch...
2    moment nscssframeconstructor attributechang ca...
3    userag mozilla5 window window nt enu rv gecko2...
4    upload extens wrong minmaxvers lengthi error m...
5    userag mozilla4 compat msie window nt sv1 net ...
6    userag mozilla5 x11 linux i686 engb rv gecko20...
7    userag mozilla5 window window nt enu rv b2 gec...
8    userag mozilla5 window window nt enu rv a1 gec...
9    testcas bug cairo build border borderradiu don...
Name: nopuntext, dtype: object

### 3. Remove Stopwords

In [6]:
nltk.download('stopwords')
from nltk.corpus import stopwords
", ".join(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


"i, me, my, myself, we, our, ours, ourselves, you, you're, you've, you'll, you'd, your, yours, yourself, yourselves, he, him, his, himself, she, she's, her, hers, herself, it, it's, its, itself, they, them, their, theirs, themselves, what, which, who, whom, this, that, that'll, these, those, am, is, are, was, were, be, been, being, have, has, had, having, do, does, did, doing, a, an, the, and, but, if, or, because, as, until, while, of, at, by, for, with, about, against, between, into, through, during, before, after, above, below, to, from, up, down, in, out, on, off, over, under, again, further, then, once, here, there, when, where, why, how, all, any, both, each, few, more, most, other, some, such, no, nor, not, only, own, same, so, than, too, very, s, t, can, will, just, don, don't, should, should've, now, d, ll, m, o, re, ve, y, ain, aren, aren't, couldn, couldn't, didn, didn't, doesn, doesn't, hadn, hadn't, hasn, hasn't, haven, haven't, isn, isn't, ma, mightn, mightn't, mustn, mus

In [7]:
STOPW=set(stopwords.words('english'))
def remove_stopwords(text):
    return " ".join([word for word in str(text).split() if word not in STOPW])
df["stopwtext"] = df["nopuntext"].apply(lambda text: remove_stopwords(text))
df["stopwtext"].head(10)

0    userag mozilla4 compat msie window nt avant br...
1    beta recreat shippedlocal file browser scratch...
2    moment nscssframeconstructor attributechang ca...
3    userag mozilla5 window window nt enu rv gecko2...
4    upload extens wrong minmaxvers lengthi error m...
5    userag mozilla4 compat msie window nt sv1 net ...
6    userag mozilla5 x11 linux i686 engb rv gecko20...
7    userag mozilla5 window window nt enu rv b2 gec...
8    userag mozilla5 window window nt enu rv a1 gec...
9    testcas bug cairo build border borderradiu ren...
Name: stopwtext, dtype: object

### 4. Remove Frequent Words

In [8]:
from collections import Counter
cnt=Counter()
for text in df["stopwtext"].values:
    for word in text.split():
        cnt[word]+=1
cnt.most_common(10)

[('gt', 248),
 ('window', 133),
 ('bug', 133),
 ('page', 93),
 ('lt', 76),
 ('attach', 74),
 ('comment', 72),
 ('firefox', 70),
 ('detail', 62),
 ('creat', 55)]

In [9]:
FREQWS=set([w for (w, wc) in cnt.most_common(101)])
def remove_freqwords(text):
    return " ".join([word for word in str(text).split() if word not in FREQWS])
df["stopftext"]=df["stopwtext"].apply(lambda text: remove_freqwords(text))
df["stopftext"].head(10)

0    mozilla4 compat msie avant avant net clr win x...
1    beta recreat shippedlocal scratch qa statu rel...
2    moment nscssframeconstructor attributechang ca...
3    correctli form xhtml selfclos tag br img src a...
4    upload wrong minmaxvers lengthi error messag e...
5    mozilla4 compat msie sv1 net clr net clr rende...
6    x11 linux i686 engb gecko20060731 ubuntudapper...
7    b2 gecko20060821 firefox2 b2 b2 gecko20060821 ...
8    a1 gecko20060808 seamonkey1 a1 gecko20060808 s...
9    testcas cairo border borderradiu render border...
Name: stopftext, dtype: object

### 5. Stem
Reducing inflected (or sometimes derived) words to their word stem from the text data.

In [10]:
from nltk.stem.porter import PorterStemmer
stemmer = PorterStemmer()
def stem_words(text):
    return " ".join([stemmer.stem(word) for word in text.split()])
df["stemtext"] = df["stopftext"].apply(lambda text: stem_words(text))
df["stemtext"].head(10)

0    mozilla4 compat msie avant avant net clr win x...
1    beta recreat shippedloc scratch qa statu relea...
2    moment nscssframeconstructor attributechang ca...
3    correctli form xhtml selfclo tag br img src al...
4    upload wrong minmaxv lengthi error messag end ...
5    mozilla4 compat msie sv1 net clr net clr rende...
6    x11 linux i686 engb gecko20060731 ubuntudapper...
7    b2 gecko20060821 firefox2 b2 b2 gecko20060821 ...
8    a1 gecko20060808 seamonkey1 a1 gecko20060808 s...
9    testca cairo border borderradiu render border ...
Name: stemtext, dtype: object

### 6. Lemmatize
Reducing inflected words to their word stem from the text data but still saving the root word (also called as lemma) belonging to the language.

In [11]:
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
lemma = WordNetLemmatizer()
def lemma_words(text):
    return " ".join([lemma.lemmatize(word) for word in text.split()])

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [12]:
df["lemmatext"] = df["stemtext"].apply(lambda text: lemma_words(text))
df["lemmatext"].head(10)

0    mozilla4 compat msie avant avant net clr win x...
1    beta recreat shippedloc scratch qa statu relea...
2    moment nscssframeconstructor attributechang ca...
3    correctli form xhtml selfclo tag br img src al...
4    upload wrong minmaxv lengthi error messag end ...
5    mozilla4 compat msie sv1 net clr net clr rende...
6    x11 linux i686 engb gecko20060731 ubuntudapper...
7    b2 gecko20060821 firefox2 b2 b2 gecko20060821 ...
8    a1 gecko20060808 seamonkey1 a1 gecko20060808 s...
9    testca cairo border borderradiu render border ...
Name: lemmatext, dtype: object

### 7. Remove URLs

In [13]:
def remove_url(text):
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'',text)
df["urltext"] = df["lemmatext"].apply(lambda text: remove_url(text))
df["urltext"].head(10)

0    mozilla4 compat msie avant avant net clr win x...
1    beta recreat shippedloc scratch qa statu relea...
2    moment nscssframeconstructor attributechang ca...
3    correctli form xhtml selfclo tag br img src al...
4    upload wrong minmaxv lengthi error messag end ...
5    mozilla4 compat msie sv1 net clr net clr rende...
6    x11 linux i686 engb gecko20060731 ubuntudapper...
7    b2 gecko20060821 firefox2 b2 b2 gecko20060821 ...
8    a1 gecko20060808 seamonkey1 a1 gecko20060808 s...
9    testca cairo border borderradiu render border ...
Name: urltext, dtype: object

### 8. Remove HTML Tags

In [14]:
def remove_html(text):
    html_pattern = re.compile("<.*?>")
    return html_pattern.sub(r'', text)
df["htmltext"] = df["urltext"].apply(lambda text: remove_html(text))
df["htmltext"].head(10)

0    mozilla4 compat msie avant avant net clr win x...
1    beta recreat shippedloc scratch qa statu relea...
2    moment nscssframeconstructor attributechang ca...
3    correctli form xhtml selfclo tag br img src al...
4    upload wrong minmaxv lengthi error messag end ...
5    mozilla4 compat msie sv1 net clr net clr rende...
6    x11 linux i686 engb gecko20060731 ubuntudapper...
7    b2 gecko20060821 firefox2 b2 b2 gecko20060821 ...
8    a1 gecko20060808 seamonkey1 a1 gecko20060808 s...
9    testca cairo border borderradiu render border ...
Name: htmltext, dtype: object

### 9. Correct Spelling Mistakes

In [15]:
!pip install pyspellchecker

You are using pip version 19.0.3, however version 21.1.1 is available.
You should consider upgrading via the 'python -m pip install --upgrade pip' command.


In [17]:
from spellchecker import SpellChecker
spell = SpellChecker()
def correct_spellings(text):
    corrected_text = []
    mispelled_word = spell.unknown(text.split())
    for word in text.split():
        if word in mispelled_word:
            corrected_text.append(spell.correction(word))
        else:
            corrected_text.append(word)
    return " ".join(corrected_text)

In [18]:
df["correcttext"] = df["htmltext"].apply(lambda text: correct_spellings(text))
df["correcttext"].head(10)

0    momzilla combat sie avant avant net car win up...
1    beta retreat shippedloc scratch qa state rela ...
2    moment nscssframeconstructor attributechang ca...
3    correctly form himl selfclo tag be im arc alt ...
4    upload wrong minimax length error message end ...
5    momzilla combat sie svu net car net car render...
6    xu linux i686 eng gecko20060731 ubuntudapperse...
7    be gecko20060821 firebox be be gecko20060821 f...
8    a gecko20060808 seamonkey1 a gecko20060808 sea...
9    testa cairo border borderradiu render border n...
Name: correcttext, dtype: object